# Fine-tune LFM2-700M on Aaron Rohrbacher Q&A → GGUF (Kaggle version)

End-to-end: train → merge → convert to GGUF → quantize → download.

**Requirements:** Kaggle notebook with GPU P100 (Settings → Accelerator → GPU P100). Free tier gives 30h/week.

**Time:** ~15-20 minutes end-to-end.

**Setup before running this notebook:**

1. **Upload your 5 JSONLs as a Kaggle Dataset:**
   - kaggle.com → Datasets → New Dataset → drag the 5 files
     (`dataset_v2.jsonl`, `dataset_v3_gap.jsonl`, `dataset_v3_projects.jsonl`,
      `dataset_v3_resume.jsonl`, `dataset_v3_adversarial.jsonl`)
   - Name it `aaron-chat-training` (or anything — if different, update `DATA_DIR` in cell 4)
   - Privacy: Private. Click Create.

2. **Attach the dataset to this notebook:**
   - Right sidebar → "+ Add Input" → search your dataset → Add
   - It mounts at `/kaggle/input/datasets/aaronrohrbacher/aaron-chat-training/`

3. **Notebook settings (right sidebar):**
   - Accelerator: **GPU P100**
   - Internet: **On** (needed to pull Unsloth base model from HF Hub)
   - Persistence: **No persistence** is fine

**Output:** `LFM2-700M-Q8_0-aaron.gguf` (~790 MB) lands in `/kaggle/working/` and is downloadable from the notebook's Output panel (top-right "Output" button) after the run completes.

## 1. Install dependencies

Unsloth handles LFM2's hybrid conv+attention architecture automatically and cuts VRAM ~50%.

In [ ]:
# Unsloth + deps. Remove %%capture if present so you can watch progress.
!pip install --upgrade pip
!pip install unsloth
# Pinned transformers/trl/peft for reproducibility
!pip install --upgrade 'transformers>=4.56' 'trl>=0.12' 'peft>=0.13' 'datasets>=3.0' 'bitsandbytes>=0.44' 'accelerate>=1.0'

## 2. Locate training data

The JSONLs should already be attached as a Kaggle Dataset at `/kaggle/input/datasets/aaronrohrbacher/aaron-chat-training/`.
If you named it something else, edit `DATA_DIR` below to match.

In [ ]:
import os

# Point at the attached Kaggle Dataset. Change if you named it something else.
DATA_DIR = '/kaggle/input/datasets/aaronrohrbacher/aaron-chat-training'

assert os.path.isdir(DATA_DIR), (
    f'{DATA_DIR} not found. Attach your dataset: right sidebar → + Add Input → search → Add. '
    f'Or edit DATA_DIR above to match your dataset name.'
)

expected = [
    'dataset_v2.jsonl',
    'dataset_v3_gap.jsonl',
    'dataset_v3_projects.jsonl',
    'dataset_v3_resume.jsonl',
    'dataset_v3_adversarial.jsonl',
]
for f in expected:
    path = os.path.join(DATA_DIR, f)
    assert os.path.isfile(path), f'Missing: {path}'
    print(f'OK: {path}')

## 3. Load LFM2-700M (4-bit) via Unsloth

Unsloth's pre-quantized build loads faster and uses ~50% less VRAM than stock.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024  # Plenty for short Q&A; training examples are well under this.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/LFM2-700M-unsloth-bnb-4bit',
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

print('Model:', model.config._name_or_path if hasattr(model.config, '_name_or_path') else type(model).__name__)
print('Chat template present:', tokenizer.chat_template is not None)

## 4. Attach LoRA adapters

Unsloth auto-detects LFM2's attention projection modules and targets them.
Convolution blocks aren't LoRA-targetable by default — that's fine for instruction tuning.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                   # LoRA rank
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'out_proj',
        'in_proj', 'w1', 'w2', 'w3',
    ],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

## 5. Load + format dataset

We prepend a system prompt to every example. The model learns to follow it at inference.

**If you change the system prompt here, change it in `src/components/ChatAgent.jsx` too** — keep training and inference identical.

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are A-A-Bot, a chat assistant on Aaron Rohrbacher's portfolio site. "
    "You know Aaron's professional background, skills, and projects.\n\n"
    "Follow these rules in order, every turn:\n\n"
    "1. READ the facts below before answering. Your answer must be supported by an explicit statement in the facts.\n"
    "2. DO NOT infer, calculate, estimate, or combine facts to produce new claims. If the facts say \"January 2018\" but don't say \"8 years of experience,\" you do NOT compute the difference — you quote what's there or decline.\n"
    "3. DO NOT fabricate. If a name, date, number, employer, project, or detail is not literally written in the facts, you do not know it. Making up plausible-sounding information is the worst thing you can do.\n"
    "4. If the facts don't answer the question, say: \"I don't have that info — just say 'connect me' and I'll open a live chat with Aaron!\"\n"
    "5. For off-topic questions (other people, philosophy, politics, current events, weather, math), briefly redirect back to Aaron.\n"
    "6. Never ask the user to upload, provide, or share any document — you already know Aaron's background.\n"
    "7. Answer briefly — one or two sentences."
)

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

background = load_jsonl(os.path.join(DATA_DIR, 'dataset_v2.jsonl'))
gap = load_jsonl(os.path.join(DATA_DIR, 'dataset_v3_gap.jsonl'))
projects = load_jsonl(os.path.join(DATA_DIR, 'dataset_v3_projects.jsonl'))
resume = load_jsonl(os.path.join(DATA_DIR, 'dataset_v3_resume.jsonl'))
adversarial = load_jsonl(os.path.join(DATA_DIR, 'dataset_v3_adversarial.jsonl'))
all_records = background + gap + projects + resume + adversarial
print(f'Total examples: {len(all_records)} '
      f'({len(background)} bg + {len(gap)} gap + {len(projects)} projects + '
      f'{len(resume)} resume + {len(adversarial)} adversarial)')

def format_example(ex):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + ex['messages']
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

formatted = [format_example(ex) for ex in all_records]
dataset = Dataset.from_list(formatted)

# Shuffle once for training order; don't double the dataset (3 epochs already covers it).
train_dataset = dataset.shuffle(seed=42)
print(f'Training set: {len(train_dataset)} examples')
print('Sample formatted text:\n', train_dataset[0]['text'][:600])

## 6. Train

3 epochs, batch 2 with grad accum 4 (effective batch size 8), linear LR schedule with 5% warmup.
Packing off (safer for short chat examples). Expect ~20-40 minutes on P100 for ~440 examples.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='/kaggle/working/outputs',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='linear',
    warmup_ratio=0.05,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    dataset_text_field='text',
    optim='adamw_8bit',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

trainer.train()

## 7. Quick sanity check

Run a couple of queries before conversion to make sure training worked.

In [ ]:
# Sanity check the trained model before merging. We intentionally do NOT call
# FastLanguageModel.for_inference(model) here — it mutates in-memory state in
# ways that can break the merge step below. Generation is slightly slower
# (~20-30s for six probes instead of ~5s) but merge stays reliable.

def ask(question):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    out = model.generate(
        inputs, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Q: {question}\nA: {text}\n')

ask('What does Aaron do?')
ask('What AWS services does he know?')
ask('How many brothers does Aaron have?')
ask('What is the meaning of life?')
ask('What is Aaron\'s wife\'s name?')
ask('Can you read his resume?')

## 8. Merge LoRA into base model

GGUF conversion needs a standalone model, not a LoRA adapter. Unsloth's `save_pretrained_merged` handles dequantization + merge in one step.

In [ ]:
import gc, os, shutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

ADAPTER_DIR = '/kaggle/working/lora-adapter'
MERGED_DIR = '/kaggle/working/lfm2-700m-aaron-merged'
BASE_MODEL = 'LiquidAI/LFM2-700M'

# 1. Save the trained LoRA adapter from the in-memory 4-bit model.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('[merge] adapter saved at', ADAPTER_DIR)

# 2. Free the 4-bit wrapped model. Unsloth's save_pretrained_merged(merged_16bit)
#    does NOT actually dequantize LFM2 — the FFN weights stay as bnb-packed uint8
#    blobs with .absmax sidecars, which convert_hf_to_gguf.py can't map.
#    So we drop the 4-bit model and reload the base in bf16 to do a real merge.
del model
try:
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# 3. Reload base in bf16 (non-quantized), attach adapter, merge.
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='cuda',
)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = peft_model.merge_and_unload()

# 4. Save clean merged model with no quantization metadata.
if os.path.isdir(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tok.save_pretrained(MERGED_DIR)

assert os.path.isfile(os.path.join(MERGED_DIR, 'config.json'))
assert any(f.endswith('.safetensors') for f in os.listdir(MERGED_DIR))
print('[merge] OK — merged model at', MERGED_DIR)
!ls -la {MERGED_DIR}

## 9. Clone and build llama.cpp

We need `convert_hf_to_gguf.py` (has native LFM2 support) and `llama-quantize`.

In [ ]:
# Clone + build llama.cpp into /tmp so thousands of build artifacts don't
# count against Kaggle's 500-file /kaggle/working output cap. ~2-5 min.
LLAMA_CPP_DIR = '/tmp/llama.cpp'
!git clone --depth=1 https://github.com/ggml-org/llama.cpp.git {LLAMA_CPP_DIR}
!pip install -r {LLAMA_CPP_DIR}/requirements/requirements-convert_hf_to_gguf.txt
!cmake -B {LLAMA_CPP_DIR}/build {LLAMA_CPP_DIR} -DGGML_CUDA=OFF -DLLAMA_BUILD_SERVER=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
!cmake --build {LLAMA_CPP_DIR}/build --target llama-quantize -j $(nproc)

## 10. Convert merged model → GGUF (F16)

In [ ]:
F16_PATH = '/kaggle/working/lfm2-700m-aaron-f16.gguf'
!python {LLAMA_CPP_DIR}/convert_hf_to_gguf.py /kaggle/working/lfm2-700m-aaron-merged --outfile {F16_PATH} --outtype f16
!ls -lh {F16_PATH}

## 11. Quantize F16 → Q8_0

Q8_0 is what the site currently loads (`LFM2-700M-Q8_0.gguf`). Near-FP16 quality, ~50% smaller file.

In [ ]:
Q8_PATH = '/kaggle/working/LFM2-700M-Q8_0-aaron.gguf'
!{LLAMA_CPP_DIR}/build/bin/llama-quantize {F16_PATH} {Q8_PATH} Q8_0
!ls -lh {Q8_PATH}

## 12. Download the quantized GGUF

Kaggle keeps files written to `/kaggle/working/` as notebook outputs. After the run completes:
- Click the **Output** tab (top-right of the Kaggle notebook)
- Find `LFM2-700M-Q8_0-aaron.gguf` (~790 MB)
- Download

**Next steps (local):**
1. Move the downloaded file to `public/models/lfm2-700m-gguf/LFM2-700M-Q8_0-aaron.gguf` in the repo
2. Update `src/components/ChatAgent.jsx` to point at the new filename
3. Update the system prompt in `ChatAgent.jsx` to match `SYSTEM_PROMPT` from step 5

In [ ]:
# Kaggle has no programmatic download API — the file is already in /kaggle/working/
# which appears under the Output tab. Final sanity-print + reminder.
import os
assert os.path.isfile(Q8_PATH), f'{Q8_PATH} missing — previous cells failed'
size_mb = os.path.getsize(Q8_PATH) / (1024 * 1024)
print(f'✓ Ready to download: {Q8_PATH}  ({size_mb:.1f} MB)')
print('Download via the Output tab (top-right of this Kaggle notebook).')